In [54]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

import optuna

In [2]:
data_path = '../src/features.csv'

In [3]:
df = pd.read_csv(data_path, index_col=0)
df.head()

,record_id,r_peak,annotation_sample,type,dist_from_annotation,anomaly,mean,std,min,max,...,median,iqr,energy,rms,skew,kurtosis,max_abs,mean_abs,waveform_length,max_diff
0,100,370,370.0,N,0.0,0,-0.317893,0.190568,-0.535,0.940,...,-0.3350,0.095,38.464150,0.370637,4.708594,24.842114,0.940,0.357714,5.350,0.325
1,100,663,662.0,N,1.0,0,-0.334286,0.173790,-0.570,0.960,...,-0.3575,0.085,39.745950,0.376762,5.330684,32.266664,0.960,0.366321,5.375,0.470
2,100,947,946.0,N,1.0,0,-0.331839,0.164789,-0.645,0.860,...,-0.3600,0.085,38.436375,0.370503,5.014945,30.068983,0.860,0.360946,5.315,0.410
3,100,1231,1231.0,N,0.0,0,-0.337000,0.155884,-0.565,0.820,...,-0.3550,0.090,38.603250,0.371307,5.159448,31.413284,0.820,0.363036,5.240,0.390
4,100,1515,1515.0,N,0.0,0,-0.325554,0.166004,-0.545,0.885,...,-0.3400,0.110,37.391925,0.365435,4.924310,29.010657,0.885,0.354696,5.225,0.390


In [5]:
id_columns = [
    'record_id',
    'r_peak',
    'annotation_sample',
    'type',
    'dist_from_annotation',
    'anomaly'
]

feature_columns = [
    col for col in df.columns
    if col not in id_columns
]

In [7]:
print(feature_columns)

['mean', 'std', 'min', 'max', 'peak_to_peak', 'median', 'iqr', 'energy', 'rms', 'skew', 'kurtosis', 'max_abs', 'mean_abs', 'waveform_length', 'max_diff']


In [49]:
record_ids = df['record_id'].unique()

train_ids, temp_ids = train_test_split(
    record_ids,
    test_size=0.4,
    random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.5,
    random_state=42
)

print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Train: 27
Validation: 9
Test: 10


In [50]:
train_df_all = df[
    df['record_id'].isin(train_ids)
].copy()

train_df = train_df_all[
    train_df_all['anomaly'] == 0
].copy()

val_df = df[
    df['record_id'].isin(val_ids)
].copy()

test_df = df[
    df['record_id'].isin(test_ids)
].copy()

In [51]:
train_df['anomaly'].value_counts()

anomaly
0    44556
Name: count, dtype: int64

In [53]:
val_df['anomaly'].value_counts()

anomaly
0    13751
1     2915
Name: count, dtype: int64

In [52]:
test_df['anomaly'].value_counts()

anomaly
0    14988
1     7040
Name: count, dtype: int64

In [55]:
X_train = train_df[feature_columns]

X_val = val_df[feature_columns]
y_val = val_df['anomaly']

X_test = test_df[feature_columns]
y_test = test_df['anomaly']

In [56]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

In [62]:
def objective(trial):
    nu = trial.suggest_float('nu', 0.001, 0.2, log=True)
    gamma = trial.suggest_float('gamma', 1e-4, 1.0, log=True)
    kernel = trial.suggest_categorical('kernel', ['rbf', 'poly', 'sigmoid'])
    tol = trial.suggest_float('tol', 1e-5, 1e-2, log=True)
    
    params = {
        'kernel': kernel,
        'nu': nu,
        'gamma': gamma,
        'tol': tol
    }
    
    if kernel == 'poly':
        params['degree'] = trial.suggest_int('degree', 2, 5)
        params['coef0'] = trial.suggest_float('coef0', -1, 1)
    
    elif kernel == 'sigmoid':
        params['coef0'] = trial.suggest_float('coef0', -1, 1)
        
    model = OneClassSVM(**params)
    model.fit(X_train_scaled)
    
    pred = model.predict(X_val_scaled)
    y_pred = np.where(
        pred == -1,
        1,
        0
    )
    
    precision = precision_score(y_val, y_pred, zero_division=0)
    recall = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    
    trial.set_user_attr('precision', precision)
    trial.set_user_attr('recall', recall)
    
    return f1

In [60]:
study = optuna.create_study(
    direction='maximize'
)

[I 2026-08-29 15:34:18,540] A new study created in memory with name: no-name-421b5482-16b0-4ba9-96e0-8bb0938d720a


In [63]:
study.optimize(
    objective,
    n_trials=30
)

[I 2026-08-29 15:34:39,397] Trial 1 finished with value: 0.07542318405828155 and parameters: {'nu': 0.034827781005028806, 'gamma': 0.0023205458170060178, 'kernel': 'sigmoid', 'tol': 0.0004830590141609036, 'coef0': -0.28617775205882023}. Best is trial 1 with value: 0.07542318405828155.
[I 2026-08-29 15:34:41,870] Trial 2 finished with value: 0.05370414768640614 and parameters: {'nu': 0.019355348018422383, 'gamma': 0.005588823071718204, 'kernel': 'sigmoid', 'tol': 7.491020092928468e-05, 'coef0': -0.8938469638061113}. Best is trial 1 with value: 0.07542318405828155.
[I 2026-08-29 15:35:15,620] Trial 3 finished with value: 0.1376464763162238 and parameters: {'nu': 0.19156500133497809, 'gamma': 0.002194231369689969, 'kernel': 'sigmoid', 'tol': 0.0012551326568590643, 'coef0': -0.016398275606032797}. Best is trial 3 with value: 0.1376464763162238.
[I 2026-08-29 15:35:17,584] Trial 4 finished with value: 0.2209737827715356 and parameters: {'nu': 0.0014611604471920859, 'gamma': 0.23060087756527

In [64]:
best_trial = study.best_trial

print("F1:", best_trial.value)
print("Precision:", best_trial.user_attrs['precision'])
print("Recall:", best_trial.user_attrs['recall'])
print("Params:", best_trial.params)

F1: 0.3494132985658409
Precision: 0.2339930151338766
Recall: 0.6895368782161235
Params: {'nu': 0.002359143674532908, 'gamma': 0.44155459078179077, 'kernel': 'rbf', 'tol': 1.68419794721345e-05}


In [66]:
best_model = OneClassSVM(**best_trial.params)
best_model.fit(X_train_scaled)

,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.44155459078179077
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",1.68419794721345e-05
,"nu nu: float, default=0.5An upper bound on the fraction of trainingerrors and a lower bound of the fraction of supportvectors. Should be in the interval (0, 1]. By default 0.5will be taken.",0.002359143674532908
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1
Name,Type,Value


In [67]:
val_pred = best_model.predict(X_val_scaled)
y_val_pred = np.where(val_pred == -1, 1, 0)

cm = confusion_matrix(y_val, y_val_pred)

print(cm)

[[7171 6580]
 [ 905 2010]]


In [68]:
val_results = val_df.copy()

val_results['prediction'] = y_val_pred

In [69]:
pd.crosstab(
    val_results['type'],
    val_results['prediction'],
    normalize='index'
)

prediction,0,1
type,,
A,0.230769,0.769231
E,0.009901,0.990099
F,0.000000,1.000000
L,0.348236,0.651764
N,0.521489,0.478511
R,0.941176,0.058824
V,0.171184,0.828816


In [70]:
type_results = val_results.groupby('type').agg(
    count=('type', 'size'),
    detected=('prediction', 'sum')
)

type_results['detection_rate'] = (
    type_results['detected']
    / type_results['count']
)

type_results

,count,detected,detection_rate
type,,,
A,13,10,0.769231
E,101,100,0.990099
F,2,2,1.000000
L,2013,1312,0.651764
N,13751,6580,0.478511
R,85,5,0.058824
V,701,581,0.828816


In [71]:
normal_val = val_results[
    val_results['type'] == 'N'
]

false_positive_rate = (
    normal_val['prediction'].mean()
)

print(false_positive_rate)

0.47851065377063484


In [42]:
pred = ocsvm.predict(X_test_scaled)
y_pred = np.where(pred == -1, 1, 0)

In [44]:
confusion_matrix(y_test, y_pred)

array([[14551,  2513],
       [ 2899,   707]])

In [47]:
recall = recall_score(y_test, y_pred)

print("Recall:", recall)

Recall: 0.19606211869107043
